PROGRESSIVE GROWING GAN (PGGAN): SCALARE VERSO L'ALTA RISOLUZIONE

Le PGGAN ci danno la strategia per scalare verso l'alta risoluzione.

- Training a stradi: fondamenta solide pertendo da 4x4pixel
- Tecnica di Fade-in: come aggiungere nuovi layer senza traumatizzare la rete
- Minibatch Standard Deviation: inventivare la diversità (strumento che utiliziamo per garantire che la nostra AI non diventi ripetitiva e noiosa).

Perchè l'alta risoluzione è così difficile per una GAN
Il programma delle GAN classiche è l'instabilità
Il discriminatore è troppo veloce a travare errore e il generatore si scoraggia, portando al fallimento dell'intera rete.

Imparare la struttura prima dei dettagli.
PGGAN propone di dividere questo problama in sotto-task più semplici.
Iniziamo addestrando il modello su immagini piccolissime di soli 4x4 pixel, dove la rete deve imparare solo la composizione globale e le palette cromatica fondamentali.

Ma come si sviluppa questa crescita?

La Piramide della Risoluzione
Crescere insieme ai dati
- Il training inzia con il Generatore e un Discriminatore semplificati per gestire una risoluzione 4x4
- Una volta stabilizzato il primo livello, si aggiunge un nuovo blocco convoluzoinale per passare a 8x8
- Questo processo si ripete raddoppiando ogni volta la dimensione spaziale fino a raggiungere i 1024x1024 pixel
- Ogni risoluzione aggiuntiva agisce come un raffinamento dei dettagli fini sovrapposti alla struttura già appresa. Il nuovo blocco non deve imparare tutto da zero, parti dalla struttura solida consolidato nel piano precedente.

Efficienza e Stabilità
Nelle fasi iniziali, il numero di pixel è minimo, permettendo un training velocissimo e una esplorazione rapida dello spazio latente (riduzione del carico).
I primi layer imparano a posizionare gli oggetti (es. un volto), evitando che la rete si perda cercando di definire texture prima di avere una forma (mappatura globale).
Evitiamo che la rete si perda nella texture della pelle quando ancora non ha capito dove vanno posizionati gli occhi.
La mappatura globale avviene quando i dati sono piccoli, il raffinamento quando i dati sono grandi.
La sfida tra G e D evole in modo incrementale, riducendo drastsicamente il rischio di mode collapse catastrofico, perchè la sfida da G e D evolve in modo naturale e inrementale.

A bassa risoluzione (4x4 o 8x8) la risoluzione dei dati è meno complessa e meno sparsa, rendendo molto più facile per il Discriminatore fornire gradienti utili al Generatore.
Questo approccio trasforma un problema di ottimizzazione impossibile in una sequenza di problemi di trasferimento di conoscenza.
Quallo che facciamo è una sequenza di problemi di transfer learning.
Trasferiamo la conoscenza della forma globale a risoluzioni più alte. Un gradino alla volta.

Ma attenzione, aggiungere un layer a freddo potrebbe distruggere tutto
Come evitiamo lo shock?

La Tecnica di Fade-in
Evitare lo shock architetturale.
Aggiungere improssisamente nuovi layer con pesi casuali a una rete già addestrata distruggerebbe istantaneamente la stabiilità del duollo avversario.
Immagina di essere ad un concerto e che un nuovo strumento inizia a suonare al massimo del volume, distruggerebbe tutti. Se aggiungiamo pesi casuali a una rete già addestrata i gradienti impazzirebbero ed il Discriminatore vincerebbe subito
Il Fade-in è il nostro registra che gestisce una dissolvenza incrociata tra la vecchia risoluzione e la nuova.
Il Fade-in è il meccanismo che permette di introdurre gradualmente la nuova risoluzione, mescolando l'output dei layer vecchi con quelli nuovi tramite un coefficiente alpha

Ma come funziona matematicamente questa dissolvenza?

Interpolazione Lineare dei Layer
La transizione indolore
Quando iniziamo una nuova fase, esempio passiamo dalla risoluzione 4x4 a quella 8x8 alpha è 0, l'output della rete è ancora quello vecchio, solo ingrandito. Man mano che il trainig procede alpha cresce verso 1 ed iniziamo a mescolare l'output dei vecchi layer con quello dei nuovi. Quando alpha arriva a 1 la transizione è completa, la rete ora pensa interamente nella nuova risoluzione, e il vecchio percorso di bypass viene rimosso, i pesi vecchi rimangono stabili, agendo da guida per quelli nuovi.
- Durante la transizione (iniziamo una nuova fase), l'output è una combinazione pesata tra il layer precedente up-samplato e il nuovo layer convoluzionale.
- Il parametro alpha cresce linearmete da 0 a 1 durante un numero prefissato di iterazioni
- Quando alpha raggiunge 1, la rete opera interamente alla nuova risoluzione e il vecchio percorso di bypass viene rimosso
- Questa tecnica, preserva i pesi già ottimizzati, agendo come una rampa di accellerazione per la nuova complessità visiva.

Questa danza deve essere perfettamente sincronizzata

Gestione dei Parametri
Il processo di fade-in deve avvenire contemporaneamente in entrambe le reti per mantenere il sistema bilanciato (sincronizzazione di G e D). G e D devono fare il fade-in insieme, se uno cresce e l'altro no, l'equilibrio di spezza.
Dopo ogni fase di fade-in lasciamo che la rete di risposi in una fase di stabilizzazione a risoluzione fissa, è qui che i nuovi pesi affinano i loro dettagli.
La fase di stabilizzazione normalizza ogni pixel nel Generatore per evitare che le magnitudini delle attivazioni esplodano durante il training progressivo.
Sebbene il modello cresce, la memoria viene gestita dinamicamente attivando solo i percorsi necessari per la risoluzione corrente.
Permettendoci di addestare modelli enormi anche su hardware non professionale

Senza il Fade-in non avremmo mai avuto i risultati del 2017. Questa tecnica permette di mantenere la corenza globale (la forma del cranio, la posizione delle orecchi), mentre iniziamo a disegnare i singoli capelli o i pori della pelle.
E' la differnza tra un immagine nitida e un volto che sembra guardarci dritto negli occhi.

Ma la nitidezza non serve a nulla se il modello genera sempre la stessa persona.

Minibatch Standard Deviation
Aumentare la variabilità dei campioni (diversità)
Uno dei pericoli delle GAN ed alta risoluzione è la perdita di diversità (il modello tende a diventare pigro): il modello potrebbe imparare a fare un unico volto perfetto ma ripetitivo.
La Minibatch Standard Deviation è una tecnica che fornisce al Discriminatore statistihe sulla varianza del batch forzando il Generatore a produrre campioni più vari.
Incentiviamo la varietà
Come facciamo a far capire ad una rete che sta diventando monotona se lei guarda una sola immagine alla volta, la risposta sta nel guardare l'intero gruppo, non il singolo individue.

Minibatch Standard Deviation
Statistiche del Gruppo
Rilevare la ripetitività
Chiediamo al discriminatore di quanto siano diverse le immagini all'interno di un batch.
- Calcoliamo la deviazione standard per ogni pixel e per ogni canale attraverso l'intero minibatch
- Questi valori vengono mediati in un unico valore scalare costante per tutto il batch
- Il valore viene concatenato come un nuovo canale di feature map aggiuntivo nel Discriminatore
Se il segnale produce immagini troppo simili, la deviazione standard serà bassa e il Discriminatore lo penalizzerà immediatamente.
Questo costringe il Generatore a diversificare la produzione per sopravvivere

Dove ci ha portato tutta questa tecnologia?
PGGAN è il nonno di StyleGAN
Comprendere il training progressivo è fondamentale, poichè è la base su cui è stata costruita la famiglia StyleGAN, lo standard ATTUALE del settore.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# SISTEMA DI PROGRESSIVE GROWING GAN (PGGAN) SEMPLIFICATO
# --------------------------------------------------------------------
# 1. Crescita Progressiva: Si inizia generando immagini a bassa risoluzione (es. 4x4)
#    e si aggiungono man mano livelli per aumentare i dettagli (8x8, 16x16, ecc.).
# 2. Alpha Fading: Quando si aggiunge un nuovo livello, non lo si usa subito al 100%.
#    Si introduce gradualmente tramite un parametro 'alpha' che varia da 0 a 1.
#    Questo permette alla rete di rimanere stabile durante la transizione.

class SimpleBlock(nn.Module):
    # Questa classe rappresenta un singolo blocco di costruzione della rete.
    # Ogni volta che la risoluzione aumenta, viene aggiunto un nuovo SimpleBlock.
    #
    # Ruolo:
    # Elabora le caratteristiche (features) dell'immagine alla risoluzione corrente.
    # Riceve in ingresso un tensore e restituisce un tensore processato.
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Primo strato convoluzionale: mantiene la dimensione spaziale 
        # (padding=1 impedisce alla convoluzione 3x3 di ridurre l'immagine)
        # lo scopo non è aumentare la risoluzione (per ora) ma raffinare le feature
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        
        # Secondo strato convoluzionale: raffina ulteriormente le features
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        
        # Funzione di attivazione LeakyReLU: permette un piccolo passaggio di gradienti negativi
        # per evitare neuroni morti.
        # x>0 -> x
        # x>0 -> 0.2 * x
        # Mentre con una ReLu classica se <0 restituisce 0
        # permtte ai gradienti di continuare a passare anche valori negativi, invece di perdersi
        self.activation = nn.LeakyReLU(0.2)

    def forward(self, x):
        # Passaggio 1: Convoluzione -> Attivazione
        x = self.conv1(x)
        x = self.activation(x)
        
        # Passaggio 2: Convoluzione -> Attivazione
        x = self.conv2(x)
        x = self.activation(x)
        
        # Restituisce le feature map elaborate
        return x

class SimpleGenerator(nn.Module):
    # Questa classe è il Generatore che crea l'immagine partendo da rumore casuale.
    # Gestisce la crescita della risoluzione e il fading tra i livelli.
    #
    # Interazione con SimpleBlock:
    # Il generatore mantiene una lista di SimpleBlock. Ogni volta che chiamiamo il metodo
    # 'grow()', un nuovo SimpleBlock viene aggiunto alla lista per gestire la nuova risoluzione.
    
    def __init__(self, z_dim=512):
        super().__init__()
        self.z_dim = z_dim
        
        # Layer Iniziale:
        # Trasforma il vettore latente (rumore) in un volume 4x4 iniziale.
        # Immaginalo come la "bozza" grezza dell'immagine.
        # parto da rumore 512 per arrivare a 512*4*4=8192
        self.input_layer = nn.Linear(z_dim, 512 * 4 * 4)
        
        # Lista dei blocchi convoluzionali (nn.ModuleList)
        # Iniziamo con un solo blocco che gestisce la risoluzione 4x4.
        # Man mano che la rete cresce, aggiungeremo blocchi qui (8x8, 16x16...).
        self.blocks = nn.ModuleList([
            SimpleBlock(512, 512)
        ])
        
        # Lista dei convertitori 'ToRGB'
        # Ogni livello di risoluzione ha bisogno del proprio strato per convertire
        # le feature interne nel formato immagine (3 canali RGB).
        # Questo ci permette di vedere l'immagine a qualsiasi stadio della crescita.
        self.to_rgb_layers = nn.ModuleList([
            nn.Conv2d(512, 3, kernel_size=1)
        ])
        
        # Stato attuale del generatore
        self.current_res_step = 0  # 0 indica che siamo alla risoluzione base (4x4)
        self.alpha = 1.0           # Parametro per il fading: 1.0 = transizione completa
                                   # Se alpha e 0.5, siamo a meta transizione tra vecchia e nuova risoluzione.

    def grow(self):
        # Aumenta la risoluzione del generatore aggiungendo nuovi strati.
        
        # Incrementa il contatore dello step di risoluzione
        self.current_res_step += 1
        
        # Calcola la nuova dimensione in pixel solo per stampa informativa
        new_res = 4 * (2 ** self.current_res_step)
        print(f"\n[SISTEMA] Crescita attivata. Nuova risoluzione target: {new_res}x{new_res}")
        
        # Logica per ridurre i canali man mano che la risoluzione aumenta
        # (per risparmiare memoria). Esempio: 512 -> 256 -> 128
        prev_channels = 512
        new_channels = 512 # In questo esempio semplificato non riduciamo i canali per chiarezza
        
        # 1. Creiamo il nuovo blocco convoluzionale per la nuova risoluzione
        new_block = SimpleBlock(prev_channels, new_channels)
        
        # 2. Lo aggiungiamo alla lista dei blocchi attivi
        self.blocks.append(new_block)
        
        # 3. Creiamo e aggiungiamo il nuovo convertitore ToRGB per questa risoluzione
        new_to_rgb = nn.Conv2d(new_channels, 3, kernel_size=1)
        self.to_rgb_layers.append(new_to_rgb)
        
        # IMPORTANTE: Resettiamo alpha a 0.0
        # Questo significa che inizialmente il nuovo livello non contribuirà all'immagine.
        # L'immagine sarà formata solo dall'upscaling del livello precedente.
        # Aumenteremo alpha gradualmente nel loop di training (qui simulato).
        self.alpha = 0.0

    def forward(self, z):
        # Questo metodo definisce il flusso dei dati attraverso la rete.
        # è qui che avviene la magia del Progressive Growing e dell'Alpha Fading.
        
        # Fase 1: Proiezione da Vettore Latente a Tensore 4x4
        # Trasformiamo il vettore di input z (dimensione [Batch, 512]) in un tensore piatto
        out = self.input_layer(z)
        # Rimodelliamo il tensore piatto in un cubo [Batch, 512, 4, 4]
        out = out.view(-1, 512, 4, 4)
        
        # Fase 2: Passaggio attraverso i blocchi 'stabili'
        # Passiamo attraverso tutti i blocchi tranne l'ultimo (che e quello in fase di fading).
        # Se siamo allo step 0, questo loop non viene eseguito.
        for i in range(self.current_res_step):
            # Elaborazione del blocco i-esimo
            out = self.blocks[i](out)
            # Upsampling: raddoppiamo la dimensione spaziale (es. 4x4 -> 8x8)
            # Usiamo 'nearest' neighbor che è semplice ed efficace per i primi test
            out = F.interpolate(out, scale_factor=2, mode='nearest')
            
        # A questo punto 'out' ha la dimensione della nuova risoluzione, ma contiene
        # ancora le informazioni elaborate dai livelli precedenti (upsamplati).
        
        # Fase 3: Gestione del Fading (Alpha Blending)
        
        # CASO A: Siamo alla risoluzione base (step 0)
        if self.current_res_step == 0:
            # Passiamo nel primo blocco ed esce subito l'RGB
            out = self.blocks[0](out)
            return self.to_rgb_layers[0](out)
        
        # CASO B: Siamo in una fase di crescita (step > 0)
        
        # Calcoliamo il percorso "VECCHIO" (Old Branch):
        # Convertiamo subito in RGB l'output proveniente dall'upsampling dei livelli precedenti.
        # Questo rappresenta l'immagine a bassa risoluzione, semplicemente ingrandita.
        old_rgb = self.to_rgb_layers[self.current_res_step - 1](out)
        
        # Calcoliamo il percorso "NUOVO" (New Branch):
        # Facciamo passare i dati attraverso il NUOVO blocco appena aggiunto.
        # Questo blocco imparera i dettagli fini della nuova risoluzione.
        new_features = self.blocks[self.current_res_step](out)
        # Convertiamo queste nuove feature in RGB
        new_rgb = self.to_rgb_layers[self.current_res_step](new_features)
        
        # Mix finale (Alpha Blending):
        # Combiniamo le due immagini in base al valore di alpha.
        # Se alpha=0: L'immagine e 100% old_rgb (solo upsample, il nuovo blocco e ignorato).
        # Se alpha=1: L'immagine e 100% new_rgb (il nuovo blocco e pienamente operativo).
        # Se alpha=0.5: L'immagine e una media tra le due.
        final_rgb = (1 - self.alpha) * old_rgb + self.alpha * new_rgb
        
        return final_rgb

# --- BLOCCO DI SIMULAZIONE ---
# Questa sezione simula l'uso della rete senza un vero addestramento,
# solo per mostrare come cambiano le dimensioni e come funziona il fading.

def test_pggan_flow():
    # Fissiamo il seme per riproducibilità
    torch.manual_seed(42)
    z_dim = 512
    batch_size = 1
    
    # Istanziamo il generatore
    print("Inizializzazione Generatore...")
    generator = SimpleGenerator(z_dim)
    
    # Creiamo un vettore di rumore casuale (input del generatore)
    z = torch.randn(batch_size, z_dim)
    
    # --- STEP 1: Risoluzione Base 4x4 ---
    print("\n--- FASE 1: Risoluzione Base (4x4) ---")
    img = generator(z)
    print(f"Output shape: {img.shape}")
    print(f"Valore Alpha: {generator.alpha} (1.0 significa stabile)")
    
    # --- STEP 2: Crescita a 8x8 ---
    print("\n--- FASE 2: Crescita a 8x8 (Inizio Fading) ---")
    # Chiamiamo grow() per aggiungere i layer 8x8
    generator.grow() 
    
    # Simuliamo il passaggio progressivo di alpha da 0 a 1
    steps_di_fading = 5
    for i in range(steps_di_fading + 1):
        # Aggiorniamo manualmente alpha (in un training vero lo farebbe il loop di training)
        generator.alpha = i / steps_di_fading
        
        # Generiamo l'immagine
        img = generator(z)
        
        print(f"Fading Step {i}: Alpha {generator.alpha:.1f} -> Dimensione Immagine {img.shape}")
        
        if i == 0:
            print("   (NOTA: Con Alpha=0, l'immagine e solo un upsample del 4x4 precedente)")
        elif i == steps_di_fading:
            print("   (NOTA: Con Alpha=1, l'immagine usa pienamente i nuovi strati 8x8)")
            
    # --- STEP 3: Crescita a 16x16 ---
    print("\n--- FASE 3: Ulteriore Crescita a 16x16 ---")
    generator.grow()
    
    # Impostiamo alpha a meta per vedere cosa succede
    generator.alpha = 0.5 
    img = generator(z)
    print(f"Test a meta fading (Alpha 0.5) -> Dimensione Immagine {img.shape}")
    print("   (L'immagine risultante e una media pesata tra l'upsample 8x8 e il nuovo 16x16)")

if __name__ == "__main__":
    test_pggan_flow()

Inizializzazione Generatore...

--- FASE 1: Risoluzione Base (4x4) ---
Output shape: torch.Size([1, 3, 4, 4])
Valore Alpha: 1.0 (1.0 significa stabile)

--- FASE 2: Crescita a 8x8 (Inizio Fading) ---

[SISTEMA] Crescita attivata. Nuova risoluzione target: 8x8
Fading Step 0: Alpha 0.0 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
   (NOTA: Con Alpha=0, l'immagine e solo un upsample del 4x4 precedente)
Fading Step 1: Alpha 0.2 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
Fading Step 2: Alpha 0.4 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
Fading Step 3: Alpha 0.6 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
Fading Step 4: Alpha 0.8 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
Fading Step 5: Alpha 1.0 -> Dimensione Immagine torch.Size([1, 3, 8, 8])
   (NOTA: Con Alpha=1, l'immagine usa pienamente i nuovi strati 8x8)

--- FASE 3: Ulteriore Crescita a 16x16 ---

[SISTEMA] Crescita attivata. Nuova risoluzione target: 16x16
Test a meta fading (Alpha 0.5) -> Dimensione Immagine 